# Pure LGBM (CV 0.94607, LB 0.94638) + three GPU models on its folds

**Fork of [najiama, "Pure LGBM Model CV 0.94607 LB 0.94638"](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94607-lb-0-94638).**
Licence: Apache 2.0, Kaggle's default for public notebooks; the original states no other licence. All feature engineering below is najiama's V3, kept verbatim, and the credits
najiama gives (cstdy, Evgeniy Dvorkin, starkhushi & Tilii, broccoli beef, Chris Deotte, Markus.JM) carry over.

| | 5-fold OOF AUC | Public LB |
|:--|:--:|:--:|
| Original (najiama V3, single LightGBM) | 0.94607 | 0.94638 |
| This fork, original LGBM re-run here | __LGBM_OOF__ | __LGBM_LB__ |
| This fork, 4-model rank blend | **__BLEND_OOF__** | **__BLEND_LB__** |

**What changed.** Nothing in the features. On the original's own `StratifiedKFold(5, random_state=42)` split and the
same per-fold triple target encoding, three models are trained next to the LightGBM: XGBoost (GPU, depth 7), CatBoost on
the TE matrix (GPU, symmetric trees), and CatBoost on the *raw* string keys with its own ordered target statistics
instead of the sklearn encoder. A Caruana hill climb over rank-normalised OOF picks the weights.

**Is the gain real?** Weights fit on folds 1-3 and scored on folds 4-5 that the selection never saw:
__HOLDOUT__.

**Runtime:** __RUNTIME__ on Kaggle T4 x2 (LightGBM on CPU as in the original, the other three on GPU).


## 1. The original, unchanged

Cells in this section are najiama's V3 code as published. The only edit is the data path: a fallback glob so the notebook finds the files under either Kaggle input layout.

In [ ]:
import time
T0 = time.time()
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e9/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e9/test.csv'
SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv'
ORIG_PATH  = '/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv' # 

import glob
def _find(p, name):
    if os.path.exists(p): return p
    return glob.glob(f'/kaggle/input/**/{name}', recursive=True)[0]
TRAIN_PATH, TEST_PATH, SUB_PATH = _find(TRAIN_PATH, 'train.csv'), _find(TEST_PATH, 'test.csv'), _find(SUB_PATH, 'sample_submission.csv')
ORIG_PATH = _find(ORIG_PATH, 'EV_Adoption_and_Range_Anxiety_Dataset.csv')
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
orig = pd.read_csv(ORIG_PATH)
submission = pd.read_csv(SUB_PATH)

In [ ]:
TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
orig[TARGET] = orig[TARGET].map({'Yes': 1, 'No': 0})

train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined.drop(columns=['Number_of_Cars_Owned'], inplace=True, errors='ignore')

cat_cols = combined.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in combined.columns if c not in cat_cols + ['id', 'is_train', TARGET]]

# Extract digits from the 10^-4 place up to the 10^3 place
digit_features = []
for c in num_cols:
    for k in range(-4, 4):
        col_name = f"{c}_digit{k}"
        combined[col_name] = (combined[c].fillna(0) // (10**k) % 10).astype('int8')
        digit_features.append(col_name)

# Add the new digit features so they get processed by your frequency/target encoders
num_cols.extend(digit_features)

# Map Original Dataset Target Means
orig_global_mean = orig[TARGET].mean()
for col in cat_cols + num_cols:
    if col in orig.columns:
        real_world_stats = orig.groupby(col, observed=False)[TARGET].mean()
        combined[f"{col}_org_mean"] = combined[col].map(real_world_stats).fillna(orig_global_mean).astype(float)

# Convert Numerics to String Categories
num_to_cat_cols = []
for col in num_cols:
    cat_name = f"{col}_cat"
    combined[cat_name] = combined[col].fillna('NaN').astype(str)
    num_to_cat_cols.append(cat_name)

# Global Frequency Encoding
all_cats = cat_cols + num_to_cat_cols
for col in all_cats:
    freq_mapping = combined[col].value_counts(normalize=True).to_dict()
    combined[f"{col}_fe"] = combined[col].map(freq_mapping).astype(float).fillna(0.0)

# The Mode Collapse Spike
combined['is_30k_spike'] = (combined['Annual_Income_USD'] == 30000.0).astype('int8')

# The Millionaire Cliff (100% buy rate region)
combined['is_millionaire_cliff'] = (combined['Annual_Income_USD'] >= 170537.0).astype('int8')

# The Dead Zone (0% buy rate region)
combined['is_dead_zone'] = ((combined['Annual_Income_USD'] >= 38000.0) & (combined['Annual_Income_USD'] <= 42000.0)).astype('int8')

# Environmental Concern Extremes
combined['is_env_hater'] = (combined['Environmental_Concern_Level'] == 1).astype('int8')

# ==========================================
# Markus's "Smooth Keys" (Binned Numerics)
# ==========================================
print("🔑 Adding Smooth Keys (Income/Commute Bins)...")
combined['income_exact_int'] = np.floor(combined['Annual_Income_USD']).astype(str)
combined['income100_floor']  = np.floor(combined['Annual_Income_USD'] / 100.0).astype(str)
combined['income1000_floor'] = np.floor(combined['Annual_Income_USD'] / 1000.0).astype(str)
combined['commute_integer']  = np.floor(combined['Daily_Commute_km']).astype(str)
# Adding these 4 new string columns to all_cats so they get Frequency and Target Encoded
all_cats.extend(['income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer'])

train = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test = combined[combined['is_train'] == 0].drop(columns=['is_train', TARGET])

#  FEATURE DROPPING
# Identify numeric columns to evaluate for correlatio (ignore strings/objects because .corr() will fail on them)
eval_cols = [c for c in train.columns if c not in ['id', TARGET] and pd.api.types.is_numeric_dtype(train[c])]

# Find perfectly correlated features (1.0 correlation)
corr_matrix = train[eval_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper_tri.columns if any(upper_tri[column] == 1.0)]

# Find constant features (only 1 unique value) in train or test
to_drop_const = [c for c in train.columns if train[c].nunique() == 1] + \
                [c for c in test.columns if test[c].nunique() == 1]

# Combine all bad features into a set to drop
DROP = set(to_drop_corr).union(set(to_drop_const))
DROP = [c for c in DROP if c not in ['id', TARGET]] # Safety check

if len(DROP) > 0:
    print(f"Dropping {len(DROP)} redundant/constant features")
    # print(f"Dropped features: {{DROP}}")
    train.drop(columns=DROP, inplace=True, errors='ignore')
    test.drop(columns=DROP, inplace=True, errors='ignore')
else:
    print("   -> No redundant features found.")
# ==========================================

FEATURES = [c for c in test.columns if c != 'id']
# Safely remove dropped columns from target encoding list
TARGET_ENCODE_COLS = [c for c in all_cats if c not in DROP] 

print(f"✅ Total Features: {len(FEATURES)}")
print(f"✅ Columns to Target Encode: {len(TARGET_ENCODE_COLS)}")

## 2. Four models on the original's folds

Same split, same encoder, same seed. Model A is najiama's LightGBM with its published parameters, so its OOF AUC is
the reproduction check. B and C read the identical TE matrix, so any gain they add comes from the learner, not from new
features. D skips the sklearn encoder and hands CatBoost the 41 raw string keys: the one model here whose encoding of
the categories differs, which is where the blend's diversity mostly comes from.


In [ ]:
import xgboost as xgb
from catboost import CatBoostClassifier
import shutil, subprocess
HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
XGB_DEV = {'device': 'cuda'} if HAS_GPU else {'device': 'cpu', 'n_jobs': -1}
CAT_DEV = {'task_type': 'GPU', 'devices': '0:1'} if HAS_GPU else {'task_type': 'CPU', 'thread_count': -1}
print('GPU available:', HAS_GPU)

Folds = 5
X = train[FEATURES]
y = train[TARGET].astype(int)
X_test = test[FEATURES]
skf = StratifiedKFold(n_splits=Folds, shuffle=True, random_state=42)   # the original's split, row for row
FOLD_ID = np.zeros(len(train), dtype=int)

MODELS = ['lgbm_orig', 'xgb', 'cat_te', 'cat_native']
OOF = {m: np.zeros(len(train)) for m in MODELS}
TEST = {m: np.zeros(len(test)) for m in MODELS}
SECONDS = {m: 0.0 for m in MODELS}
BEST_IT = {m: [] for m in MODELS}

def add_te(X_train, y_train, X_valid, X_test_fold):
    """najiama's triple TargetEncoder, verbatim in effect: auto / 10 / 100, fit inside the outer fold."""
    X_train, X_valid, X_test_fold = X_train.copy(), X_valid.copy(), X_test_fold.copy()
    for tag, sm in [('auto', 'auto'), ('10', 10.0), ('100', 100.0)]:
        te = TargetEncoder(shuffle=True, cv=Folds, smooth=sm, random_state=42)
        a = te.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
        b = te.transform(X_valid[TARGET_ENCODE_COLS])
        c = te.transform(X_test_fold[TARGET_ENCODE_COLS])
        for i, col in enumerate(TARGET_ENCODE_COLS):
            X_train[f"{col}_TE_{tag}"] = a[:, i].astype('float32')
            X_valid[f"{col}_TE_{tag}"] = b[:, i].astype('float32')
            X_test_fold[f"{col}_TE_{tag}"] = c[:, i].astype('float32')
    for d in (X_train, X_valid, X_test_fold):
        d.drop(columns=TARGET_ENCODE_COLS, inplace=True)
    return X_train, X_valid, X_test_fold

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    FOLD_ID[valid_idx] = fold
    X_tr_raw, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_va_raw, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]
    X_train, X_valid, X_test_fold = add_te(X_tr_raw, y_train, X_va_raw, X_test)

    # (A) the original LightGBM, same parameters, same seed
    t = time.time()
    clf = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.02, max_depth=5, num_leaves=32, min_child_samples=10,
                             subsample=0.812763, colsample_bytree=0.30293, reg_alpha=0.07094, reg_lambda=2.03303,
                             max_bin=1024, random_state=42, feature_pre_filter=False, metric='auc', n_jobs=-1, verbose=-1)
    clf.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],
            callbacks=[lgb.early_stopping(stopping_rounds=500, verbose=False)])
    OOF['lgbm_orig'][valid_idx] = clf.predict_proba(X_valid)[:, 1]
    TEST['lgbm_orig'] += clf.predict_proba(X_test_fold)[:, 1] / Folds
    BEST_IT['lgbm_orig'].append(clf.best_iteration_); SECONDS['lgbm_orig'] += time.time() - t

    # (B) XGBoost on the same TE matrix, GPU, deeper trees than the LightGBM
    t = time.time()
    xg = xgb.XGBClassifier(n_estimators=20000, learning_rate=0.02, max_depth=7, min_child_weight=5, subsample=0.8,
                           colsample_bytree=0.4, reg_lambda=3.0, reg_alpha=0.1, max_bin=512, tree_method='hist', **XGB_DEV,
                           eval_metric='auc', early_stopping_rounds=500, random_state=42)
    xg.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    OOF['xgb'][valid_idx] = xg.predict_proba(X_valid)[:, 1]
    TEST['xgb'] += xg.predict_proba(X_test_fold)[:, 1] / Folds
    BEST_IT['xgb'].append(xg.best_iteration); SECONDS['xgb'] += time.time() - t

    # (C) CatBoost on the same TE matrix, GPU: symmetric trees are a different inductive bias
    t = time.time()
    cb = CatBoostClassifier(iterations=20000, learning_rate=0.06, depth=7, l2_leaf_reg=3, border_count=254,
                            **CAT_DEV, eval_metric='AUC', od_type='Iter', od_wait=300,
                            random_seed=42, verbose=0)
    cb.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True)
    OOF['cat_te'][valid_idx] = cb.predict_proba(X_valid)[:, 1]
    TEST['cat_te'] += cb.predict_proba(X_test_fold)[:, 1] / Folds
    BEST_IT['cat_te'].append(cb.get_best_iteration()); SECONDS['cat_te'] += time.time() - t

    # (D) CatBoost on the RAW string keys: its own ordered target statistics replace the sklearn TE entirely
    t = time.time()
    cats = [c for c in TARGET_ENCODE_COLS]
    cn = CatBoostClassifier(iterations=20000, learning_rate=0.06, depth=7, l2_leaf_reg=5, max_ctr_complexity=1,
                            **CAT_DEV, eval_metric='AUC', od_type='Iter', od_wait=300,
                            random_seed=42, verbose=0)
    cn.fit(X_tr_raw, y_train, cat_features=cats, eval_set=(X_va_raw, y_valid), use_best_model=True)
    OOF['cat_native'][valid_idx] = cn.predict_proba(X_va_raw)[:, 1]
    TEST['cat_native'] += cn.predict_proba(X_test)[:, 1] / Folds
    BEST_IT['cat_native'].append(cn.get_best_iteration()); SECONDS['cat_native'] += time.time() - t

    print(f"fold {fold} | " + " | ".join(f"{m} {roc_auc_score(y_valid, OOF[m][valid_idx]):.5f}" for m in MODELS)
          + f" | elapsed {(time.time()-T0)/60:.1f} min", flush=True)
    np.save('oof_all.npy', np.vstack([OOF[m] for m in MODELS])); np.save('test_all.npy', np.vstack([TEST[m] for m in MODELS]))

print()
for m in MODELS:
    print(f"{m:15s} OOF AUC {roc_auc_score(y, OOF[m]):.5f} | best iters {BEST_IT[m]} | {SECONDS[m]/60:.1f} min")


## 3. Hill climb over the out-of-fold columns

AUC reads only the order of predictions, so each column is rank-normalised before blending. Forward selection with
replacement (Caruana 2004) adds the column that raises OOF AUC most until nothing does; the weights are the counts.
The first block fits the weights on three folds and scores two held-out folds, so the delta it prints is not chosen on
the rows it is measured on.


In [ ]:
from scipy.stats import rankdata

def ranks(v):
    return rankdata(v) / len(v)

R_OOF = np.vstack([ranks(OOF[m]) for m in MODELS]).T      # (n, 4): AUC only sees order, so blend in rank space
R_TEST = np.vstack([ranks(TEST[m]) for m in MODELS]).T

print("Spearman correlation of OOF columns (lower = more to gain from blending)")
print(pd.DataFrame(np.corrcoef(R_OOF.T), index=MODELS, columns=MODELS).round(4))

def hill_climb(R, yv, iters=60):
    """Caruana forward selection with replacement: add whichever column raises OOF AUC most; weights = counts."""
    counts = np.zeros(R.shape[1]); blend = np.zeros(len(R)); k = 0; best = -1.0
    for _ in range(iters):
        scores = [roc_auc_score(yv, (blend * k + R[:, j]) / (k + 1)) for j in range(R.shape[1])]
        j = int(np.argmax(scores))
        if scores[j] <= best + 1e-7 and k > 0:
            break
        best = scores[j]; blend = (blend * k + R[:, j]) / (k + 1); counts[j] += 1; k += 1
    return counts / counts.sum(), best

# Honest check first: fit the weights on folds 1-3, score on folds 4-5 that the selection never saw.
sel, hold = np.isin(FOLD_ID, [1, 2, 3]), np.isin(FOLD_ID, [4, 5])
w_sel, _ = hill_climb(R_OOF[sel], y.values[sel])
auc_hold_blend = roc_auc_score(y.values[hold], R_OOF[hold] @ w_sel)
auc_hold_lgbm = roc_auc_score(y.values[hold], R_OOF[hold, 0])
print("\nweights fit on folds 1-3:", {m: round(float(w), 3) for m, w in zip(MODELS, w_sel)})
print(f"held-out folds 4-5 | original LGBM {auc_hold_lgbm:.5f} | blend {auc_hold_blend:.5f} | delta {auc_hold_blend-auc_hold_lgbm:+.5f}")

# Final weights on all five folds.
W, auc_blend = hill_climb(R_OOF, y.values)
auc_lgbm = roc_auc_score(y, OOF['lgbm_orig'])
print("\nfinal weights:", {m: round(float(w), 3) for m, w in zip(MODELS, W)})
print(f"5-fold OOF AUC | original LGBM {auc_lgbm:.5f} | blend {auc_blend:.5f} | delta {auc_blend-auc_lgbm:+.5f}")

submission = pd.DataFrame({'id': test['id'].values, TARGET: R_TEST @ W})
submission.to_csv('submission.csv', index=False)
pd.DataFrame({'id': test['id'].values, TARGET: TEST['lgbm_orig']}).to_csv('submission_lgbm_orig.csv', index=False)
oof_out = pd.DataFrame({'id': train['id'].values, 'fold': FOLD_ID, **{m: OOF[m] for m in MODELS}})
oof_out.to_csv('oof_4models.csv', index=False)
pd.DataFrame({'id': test['id'].values, **{m: TEST[m] for m in MODELS}}).to_csv('test_4models.csv', index=False)
print(f"\nsaved submission.csv (blend), submission_lgbm_orig.csv, oof_4models.csv, test_4models.csv")
print(f"total runtime {(time.time()-T0)/60:.1f} min")
